In [ ]:

!pip -q install "web3>=7,<8" pandas pillow



In [ ]:
import json
import os
from google.colab import drive
from pathlib import Path

# Mount Google Drive if it is not already mounted
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")

# Permanent ABI file location in Google Drive
ABI_PATH = Path(
    "/content/drive/MyDrive/capstone_blockchain_files/ImageVerification_ABI.json"
)

# Confirm the file exists before loading it
if not ABI_PATH.is_file():
    raise FileNotFoundError(
        f"ABI file not found at: {ABI_PATH}\n"
        "Check that the file is uploaded to the correct Google Drive folder."
    )

with open(ABI_PATH, "r") as file:
    abi_data = json.load(file)

# Support both a raw ABI list and a JSON artifact containing "abi"
ABI = abi_data["abi"] if isinstance(abi_data, dict) else abi_data

print("ABI loaded successfully!")
print("ABI path:", ABI_PATH)
print("\nAvailable functions:")

for item in ABI:
    if item.get("type") == "function":
        print("-", item["name"])


In [ ]:

# ============================================================
# STEP 3: INSTALL WEB3.PY
# Project: Academic Certificate Tampering Detection
#
# Purpose:
# Web3.py allows Python/Google Colab to communicate with an
# Ethereum-compatible blockchain and call Solidity functions.
#
# Important:
# This cell only installs a Python library. It does not connect
# to the blockchain, deploy a contract, or send a transaction.
# ============================================================

%pip install "web3>=7,<8"



In [ ]:

# ============================================================
# STEP 3A: CHECK WEB3.PY INSTALLATION
#
# Purpose:
# Confirm that Web3.py was installed correctly before we
# configure the blockchain connection.
# ============================================================

import web3

print("Web3.py installed successfully!")
print("Installed version:", web3.__version__)


In [ ]:

# STEP 5C: SAVE THE CORRECT ALCHEMY ENDPOINT
# Paste the complete URL copied from the Ethereum Sepolia app.
# The URL is hidden while you enter it.

from getpass import getpass
from urllib.parse import urlparse

SEPOLIA_RPC_URL = getpass("Paste the complete Sepolia endpoint: ").strip()

# Check the URL format without revealing the API key.
parsed_url = urlparse(SEPOLIA_RPC_URL)

assert parsed_url.scheme == "https", "URL must start with https://"
assert parsed_url.hostname == "eth-sepolia.g.alchemy.com", (
    "Please copy the Ethereum Sepolia endpoint from Alchemy."
)
assert parsed_url.path.startswith("/v2/"), (
    "The endpoint path should start with /v2/."
)
assert len(parsed_url.path.removeprefix("/v2/")) > 10, (
    "The API key appears to be missing."
)

print("Endpoint format looks correct!")


In [ ]:

# ============================================================
# STEP 6: TEST THE SEPOLIA BLOCKCHAIN CONNECTION
# Project: Academic Certificate Tampering Detection
#
# Purpose:
# 1. Connect Web3.py to the Alchemy Sepolia endpoint.
# 2. Check that the endpoint responds.
# 3. Confirm that the network is Ethereum Sepolia.
# 4. Display the latest block number.
#
# This is a read-only test. It does not deploy a contract
# or send a transaction.
# ============================================================

from web3 import Web3

# Use the endpoint configured in Step 5C.
# Do not print this URL because it contains your API key.
w3 = Web3(Web3.HTTPProvider(SEPOLIA_RPC_URL, request_kwargs={"timeout": 20}))

# Check whether Web3.py can communicate with Alchemy.
if not w3.is_connected():
    raise ConnectionError(
        "Connection failed. Check your Alchemy endpoint and API key."
    )

# Get the blockchain network ID.
chain_id = w3.eth.chain_id

print("RPC connection successful!")
print("Chain ID:", chain_id)

# Sepolia's official chain ID is 11155111.
if chain_id != 11155111:
    raise RuntimeError(
        "Incorrect network. Please select Ethereum Sepolia."
    )

# Read the latest block number to confirm the endpoint responds.
latest_block = w3.eth.block_number

print("Connected to Ethereum Sepolia successfully!")
print("Latest block:", latest_block)


In [ ]:
# Install Web3.py so Python can communicate with Ethereum Sepolia
%pip install "web3>=7,<8"

In [ ]:
# Verify that Web3.py is available
import web3

print("Web3.py installed successfully!")
print("Version:", web3.__version__)

In [ ]:
# Restore the Sepolia RPC connection securely
from getpass import getpass
from urllib.parse import urlparse
from web3 import Web3

# Paste your complete Alchemy Ethereum Sepolia endpoint when prompted.
# It will be hidden while you type or paste it.
SEPOLIA_RPC_URL = getpass("Paste your Alchemy Sepolia endpoint: ").strip()

# Check the endpoint format without printing your API key.
parsed_url = urlparse(SEPOLIA_RPC_URL)

assert parsed_url.scheme == "https", "Endpoint must start with https://"
assert parsed_url.hostname == "eth-sepolia.g.alchemy.com", (
    "Use your Ethereum Sepolia endpoint from Alchemy."
)
assert parsed_url.path.startswith("/v2/"), (
    "Endpoint should contain /v2/ followed by your API key."
)

# Test the connection and confirm the network.
w3 = Web3(
    Web3.HTTPProvider(
        SEPOLIA_RPC_URL,
        request_kwargs={"timeout": 20}
    )
)

assert w3.is_connected(), "RPC connection failed. Check the endpoint."
assert w3.eth.chain_id == 11155111, "Connected to the wrong network."

print("Sepolia RPC connection restored successfully!")
print("Chain ID:", w3.eth.chain_id)
print("Latest block:", w3.eth.block_number)

In [ ]:
# Check whether the ABI file survived the runtime reset
import os

abi_path = "/content/ImageVerification_ABI.json"

if os.path.exists(abi_path):
    print("ABI file found!")
else:
    print("ABI file is missing. Please upload it again.")

In [ ]:
# Step 2: Connect to the deployed ImageVerification contract

import json
from web3 import Web3

# Deployed contract address from our Remix deployment
CONTRACT_ADDRESS = Web3.to_checksum_address(
    "0x6b28d606dab359cc612914fb60aceaaF25c46C68"
)

# Load the ABI from the file in Colab
with open(ABI_PATH, "r") as f:
    abi_data = json.load(f)

# Support both a raw ABI list and a JSON artifact containing "abi"
contract_abi = abi_data["abi"] if isinstance(abi_data, dict) else abi_data

# Connect Python to the already deployed smart contract
contract = w3.eth.contract(
    address=CONTRACT_ADDRESS,
    abi=contract_abi
)

# Check that bytecode exists at this address on Sepolia
deployed_code = w3.eth.get_code(CONTRACT_ADDRESS)
assert len(deployed_code) > 0, (
    "No contract code found at this address. Check the address on Sepolia."
)

print("Contract connected successfully!")
print("Contract address:", CONTRACT_ADDRESS)
print("Contract code size:", len(deployed_code), "bytes")

In [ ]:
# Step 3: Read the owner of the deployed smart contract
# This is a read-only call, so it does not cost gas.

owner_address = contract.functions.owner().call()

print("Contract owner:", owner_address)
print("Connected contract:", CONTRACT_ADDRESS)

# Check whether the returned owner is a valid Ethereum address
assert Web3.is_address(owner_address), "Invalid owner address returned"

print("Owner function executed successfully!")

In [ ]:
# Step 5: Prepare a sample image record for testing
# This creates a sample hash; it does not represent a real certificate.

import hashlib

# Unique ID for this test record
test_image_id = "IMG_TEST_001"

# Generate a sample SHA-256 digest from sample text
sample_data = b"Sample certificate for blockchain integration test"
test_hash_hex = hashlib.sha256(sample_data).hexdigest()

# Convert the 64-character hexadecimal digest into bytes32
test_hash_bytes = bytes.fromhex(test_hash_hex)

# Example prediction and confidence
test_prediction = "Original"
test_confidence_bps = 9640  # 96.40%

print("Image ID:", test_image_id)
print("SHA-256:", test_hash_hex)
print("Hash length:", len(test_hash_bytes), "bytes")
print("Prediction:", test_prediction)
print("Confidence:", test_confidence_bps / 100, "%")

assert len(test_hash_bytes) == 32, "SHA-256 must be exactly 32 bytes"
print("Sample record is ready.")

In [ ]:
# Step 6: Register a sample record on Ethereum Sepolia
# This sends a real transaction on the test network and uses test ETH for gas.

from getpass import getpass
from web3 import Web3

# Enter the private key of your dedicated test wallet.
# The input is hidden and should never be printed or shared.
private_key = getpass("Enter your TEST wallet private key: ").strip()

# Derive the sending account from the private key.
account = w3.eth.account.from_key(private_key)

# Safety check: only the contract owner should register this record.
owner_address = contract.functions.owner().call()
if account.address.lower() != owner_address.lower():
    private_key = None
    raise ValueError(
        "Wallet does not match the contract owner. "
        "No transaction was sent."
    )

# Confirm the wallet has enough Sepolia ETH for gas.
balance_wei = w3.eth.get_balance(account.address)
print("Owner wallet:", account.address)
print("Sepolia balance:", w3.from_wei(balance_wei, "ether"), "ETH")

if balance_wei == 0:
    private_key = None
    raise ValueError("Wallet has no Sepolia ETH for gas.")

# Prepare the registration transaction using the actual contract function.
# The SHA-256 digest must be bytes32, not a text string.
tx = contract.functions.registerImage(
    test_image_id,
    test_hash_bytes,
    test_prediction,
    test_confidence_bps
).build_transaction({
    "from": account.address,
    "nonce": w3.eth.get_transaction_count(account.address, "pending"),
    "chainId": 11155111,
    "gas": 200000,
    "gasPrice": w3.eth.gas_price
})

# Sign the transaction locally and submit it to Sepolia.
signed_tx = account.sign_transaction(tx)
tx_hash = w3.eth.send_raw_transaction(signed_tx.raw_transaction)

# Clear references to the private key and signed transaction where possible.
private_key = None
signed_tx = None

print("Registration transaction submitted.")
print("Transaction hash:", tx_hash.hex())

# Wait for the blockchain to confirm the transaction.
receipt = w3.eth.wait_for_transaction_receipt(tx_hash, timeout=180)

if receipt.status != 1:
    raise RuntimeError("Registration transaction failed.")

print("Registration confirmed!")
print("Block number:", receipt.blockNumber)
print("Gas used:", receipt.gasUsed)
print("Status:", receipt.status)

In [ ]:
record = contract.functions.getImageRecord("IMG_TEST_001").call()

print("Record exists:", record[4])
print("SHA-256 hash:", record[0].hex())
print("Prediction:", record[1])
print("Confidence:", record[2] / 100, "%")
print("Timestamp:", record[3])

In [ ]:
!pip -q install web3

In [ ]:
from web3 import Web3
from getpass import getpass

rpc_url = getpass("Enter your Alchemy Sepolia RPC URL: ")
w3 = Web3(Web3.HTTPProvider(rpc_url))

print("Connected:", w3.is_connected())
print("Chain ID:", w3.eth.chain_id if w3.is_connected() else "Not connected")

In [ ]:
import json
from web3 import Web3

CONTRACT_ADDRESS = Web3.to_checksum_address(
    "0x6b28D606DAB359cc612914fB60AceAaf25c46C68"
)

with open("/content/ImageVerification_ABI.json", "r") as f:
    abi_data = json.load(f)

contract_abi = (
    abi_data["abi"] if isinstance(abi_data, dict) else abi_data
)

contract = w3.eth.contract(
    address=CONTRACT_ADDRESS,
    abi=contract_abi
)

print("Contract connected:", contract.address)
print("Contract code exists:", len(w3.eth.get_code(CONTRACT_ADDRESS)) > 0)
print("Owner:", contract.functions.owner().call())

In [ ]:
record = contract.functions.getImageRecord("IMG_TEST_001").call()

print("Record exists:", record[4])
print("SHA-256 hash:", record[0].hex())
print("Prediction:", record[1])
print("Confidence:", record[2] / 100, "%")
print("Timestamp:", record[3])

In [ ]:
print("Contract owner:", contract.functions.owner().call())
print("Connected wallet:", Web3.to_checksum_address(
    "0xCee4832bb704505b8Bd3c672CaEbb34FA0cE78D8"
))
print("Same address:",
      contract.functions.owner().call().lower() ==
      "0xCee4832bb704505b8Bd3c672CaEbb34FA0cE78D8".lower())

In [ ]:
print("Contract address:", contract.address)
print("Owner:", contract.functions.owner().call())
print("Latest Sepolia block:", w3.eth.block_number)
print("Registration function available:",
      hasattr(contract.functions, "registerImage"))

In [ ]:
test_image_id = "IMG_TEST_001"
test_hash_hex = "f4b3a3fdbc66f908ad75e00c7e0288217312cf494d022722b1d56b4891a5ec4e"
test_hash_bytes = bytes.fromhex(test_hash_hex)
test_prediction = "Original"
test_confidence_bps = 9640

try:
    gas_estimate = contract.functions.registerImage(
        test_image_id,
        test_hash_bytes,
        test_prediction,
        test_confidence_bps
    ).estimate_gas({
        "from": contract.functions.owner().call()
    })
    print("Simulation succeeded.")
    print("Estimated gas:", gas_estimate)
except Exception as e:
    print("Simulation failed:")
    print(e)


In [ ]:
from getpass import getpass
from eth_account import Account

# Use the sample values already prepared above.
owner = contract.functions.owner().call()

# Enter the dedicated test wallet's private key securely.
private_key = getpass("Enter test wallet private key (hidden): ")
account = Account.from_key(private_key)

if account.address.lower() != owner.lower():
    private_key = None
    raise ValueError("This wallet is not the contract owner. No transaction sent.")

tx = contract.functions.registerImage(
    test_image_id,
    test_hash_bytes,
    test_prediction,
    test_confidence_bps
).build_transaction({
    "from": account.address,
    "nonce": w3.eth.get_transaction_count(account.address, "pending"),
    "chainId": 11155111,
    "gas": 700000,
    "gasPrice": w3.eth.gas_price
})

signed_tx = account.sign_transaction(tx)
tx_hash = w3.eth.send_raw_transaction(signed_tx.raw_transaction)

# Clear local references to the secret as soon as possible.
private_key = None
signed_tx = None
account = None

print("Transaction submitted.")
print("Transaction hash:", tx_hash.hex())
print("Waiting for confirmation...")

receipt = w3.eth.wait_for_transaction_receipt(tx_hash, timeout=180)
print("Transaction status:", receipt.status)
print("Gas used:", receipt.gasUsed)
print("Block number:", receipt.blockNumber)

In [ ]:
record = contract.functions.getImageRecord("IMG_TEST_001").call()

print("Record exists:", record[4])
print("SHA-256 hash:", "0x" + record[0].hex())
print("Prediction:", record[1])
print("Confidence:", record[2] / 100, "%")
print("Timestamp:", record[3])

In [ ]:
# Test 1: Verify the same hash
same_result = contract.functions.verifyImage(
    "IMG_TEST_001",
    test_hash_bytes
).call()

print("Same hash test")
print("Registered:", same_result[0])
print("Hash matches:", same_result[1])

# Test 2: Verify a different hash
modified_hash = bytes.fromhex(
    "a4b3a3fdbc66f908ad75e00c7e0288217312cf494d022722b1d56b4891a5ec4e"
)

modified_result = contract.functions.verifyImage(
    "IMG_TEST_001",
    modified_hash
).call()

print("\nModified hash test")
print("Registered:", modified_result[0])
print("Hash matches:", modified_result[1])

In [ ]:

!pip -q install fastapi uvicorn




from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI(title="Certificate Verification API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=[
        "http://localhost:5173",
        "http://127.0.0.1:5173"
    ],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/")
def home():
    return {"message": "Certificate Verification API is running"}

@app.get("/health")
def health():
    return {"status": "ok"}



In [ ]:
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from hashlib import sha256
from datetime import datetime, timezone

app = FastAPI(title="Certificate Verification API")

# Allow the React frontend on Vite's common development ports
app.add_middleware(
    CORSMiddleware,
    allow_origins=[
        "http://localhost:5173",
        "http://127.0.0.1:5173",
        "http://localhost:5174",
        "http://127.0.0.1:5174",
    ],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# Temporary in-memory history. Records disappear when this server restarts.
verification_history = []

@app.get("/")
def home():
    return {"message": "Certificate Verification API is running"}

@app.get("/health")
@app.get("/api/health")
def health():
    return {"status": "ok"}

@app.get("/api/contract-info")
def contract_info():
    # Blockchain integration will be configured in a later step.
    return {
        "configured": False,
        "network": "Sepolia",
        "message": "Smart contract connection not configured yet"
    }

@app.get("/api/history")
def history():
    return {"history": verification_history}

@app.post("/api/verify")
async def verify_certificate(
    file: UploadFile = File(...),
    certificate_id: str = Form("")
):
    allowed_types = {
        "image/png",
        "image/jpeg",
        "application/pdf"
    }

    if file.content_type not in allowed_types:
        raise HTTPException(
            status_code=400,
            detail="Upload a PNG, JPG, JPEG, or PDF certificate."
        )

    content = await file.read()

    if not content:
        raise HTTPException(status_code=400, detail="The uploaded file is empty.")

    if len(content) > 10 * 1024 * 1024:
        raise HTTPException(
            status_code=400,
            detail="File exceeds the 10 MB limit."
        )

    file_hash = sha256(content).hexdigest()

    record = {
        "certificate_id": certificate_id or "Not provided",
        "filename": file.filename,
        "sha256": file_hash,
        "verified_at": datetime.now(timezone.utc).isoformat(),
    }

    verification_history.insert(0, record)

    # Important: AI inference and blockchain verification are not
    # implemented here, so do not report the certificate as genuine.
    return {
        "certificate_id": record["certificate_id"],
        "filename": file.filename,
        "sha256": file_hash,
        "file_hash": file_hash,
        "ai": {
            "status": "not_configured",
            "message": "AI tampering detection model is not connected yet"
        },
        "blockchain": {
            "status": "not_configured",
            "message": "Smart contract verification is not connected yet"
        },
        "message": "File received and SHA-256 hash calculated. Authenticity has not been verified."
    }

In [ ]:
!pip install fastapi uvicorn python-multipart pyngrok

In [ ]:
from pyngrok import ngrok

# Open a tunnel to the FastAPI server on port 8000
tunnel = ngrok.connect(8000, "http")

print("YOUR NEW BACKEND URL:")
print(tunnel.public_url)

In [ ]:
print("App title:", app.title)
print("Registered routes:")

for route in app.routes:
    print(getattr(route, "methods", None), route.path)

In [ ]:
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from hashlib import sha256
from datetime import datetime, timezone

app = FastAPI(title="Certificate Verification API")

app.add_middleware(
    CORSMiddleware,
    allow_origins=[
        "http://localhost:5173",
        "http://127.0.0.1:5173",
        "http://localhost:5174",
        "http://127.0.0.1:5174",
    ],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

verification_history = []

@app.get("/")
def home():
    return {"message": "Certificate Verification API is running"}

@app.get("/health")
@app.get("/api/health")
def health():
    return {"status": "ok"}

@app.get("/api/contract-info")
def contract_info():
    return {
        "configured": False,
        "network": "Sepolia",
        "message": "Smart contract connection not configured yet"
    }

@app.get("/api/history")
def history():
    return {"history": verification_history}

@app.post("/api/verify")
async def verify_certificate(
    file: UploadFile = File(...),
    certificate_id: str = Form("")
):
    allowed_types = {"image/png", "image/jpeg", "application/pdf"}

    if file.content_type not in allowed_types:
        raise HTTPException(
            status_code=400,
            detail="Upload a PNG, JPG, JPEG, or PDF certificate."
        )

    content = await file.read()

    if not content:
        raise HTTPException(status_code=400, detail="The uploaded file is empty.")

    if len(content) > 10 * 1024 * 1024:
        raise HTTPException(status_code=400, detail="File exceeds the 10 MB limit.")

    file_hash = sha256(content).hexdigest()

    record = {
        "certificate_id": certificate_id or "Not provided",
        "filename": file.filename,
        "sha256": file_hash,
        "verified_at": datetime.now(timezone.utc).isoformat(),
    }

    verification_history.insert(0, record)

    return {
        "certificate_id": record["certificate_id"],
        "filename": file.filename,
        "sha256": file_hash,
        "file_hash": file_hash,
        "ai": {
            "status": "not_configured",
            "message": "AI tampering detection model is not connected yet"
        },
        "blockchain": {
            "status": "not_configured",
            "message": "Smart contract verification is not connected yet"
        },
        "message": "File received and SHA-256 hash calculated. Authenticity has not been verified."
    }

In [ ]:
import threading
import uvicorn

def start_api():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

api_thread = threading.Thread(target=start_api, daemon=True)
api_thread.start()

In [ ]:
from pyngrok import ngrok

ngrok.kill()

tunnel = ngrok.connect(8000, "http")
print("NEW BACKEND URL:", tunnel.public_url)

In [ ]:
# Step 1: Upload the CSV or Excel file containing original certificate hashes

from google.colab import files
import pandas as pd
import io

uploaded = files.upload()

# Get the uploaded filename
filename = next(iter(uploaded))

# Read CSV or Excel based on file extension
if filename.lower().endswith(".csv"):
    df = pd.read_csv(io.BytesIO(uploaded[filename]))
elif filename.lower().endswith((".xlsx", ".xls")):
    df = pd.read_excel(io.BytesIO(uploaded[filename]))
else:
    raise ValueError("Please upload a .csv, .xlsx, or .xls file.")

# Display column names and first few rows
print("Columns found:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nTotal records:", len(df))

In [ ]:

!pip install -q pyngrok


In [ ]:

from getpass import getpass
from pyngrok import ngrok

token = getpass("Paste your ngrok authtoken: ")
ngrok.set_auth_token(token)

tunnel = ngrok.connect(8000, "http")
public_url = tunnel.public_url

print("Your backend URL:", public_url)


In [ ]:

@app.get("/health")
def health():
    return {"status": "ok"}


In [ ]:

import threading
import uvicorn

def start_api():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

api_thread = threading.Thread(target=start_api, daemon=True)
api_thread.start()


In [ ]:
!pip install fastapi uvicorn python-multipart pyngrok


In [ ]:
import pandas as pd
import re

# Load the CSV uploaded in the previous step
df = pd.read_csv("image_hashes (1).csv")

# Select the first original certificate
row = df.iloc[0]

image_id = str(row["image_id"]).strip()
filename = str(row["filename"]).strip()
sha256_hex = str(row["sha256"]).strip().lower()

# Validate the SHA-256 hash: exactly 64 hexadecimal characters
if not re.fullmatch(r"[0-9a-f]{64}", sha256_hex):
    raise ValueError(
        f"Invalid SHA-256 hash for {image_id}. "
        "Expected exactly 64 hexadecimal characters without 0x."
    )

# Convert the hash to the 32-byte value required by the Solidity contract
sha256_bytes = bytes.fromhex(sha256_hex)

print("Certificate ID:", image_id)
print("Filename:", filename)
print("SHA-256:", sha256_hex)
print("Hash length:", len(sha256_bytes), "bytes")
print("Ready for registration:", len(sha256_bytes) == 32)

In [ ]:
# Register the actual original certificate IMG_0002.
# First, simulate the transaction without sending it.

image_id = "IMG_0002"
sha256_hex = (
    "ebb5bb8fd45137c2b58ae25c32a42ef03c61f3119bc37cfc9d7dd018a2635b24"
)
sha256_bytes = bytes.fromhex(sha256_hex)

prediction = "Original"
confidence_bps = 9793

# Verify that the hash has the required 32-byte length.
assert len(sha256_bytes) == 32, "Invalid SHA-256 length"

# Simulate registration from the contract owner.
owner = contract.functions.owner().call()

gas_estimate = contract.functions.registerImage(
    image_id,
    sha256_bytes,
    prediction,
    confidence_bps
).estimate_gas({"from": owner})

print("Image ID:", image_id)
print("Hash length:", len(sha256_bytes), "bytes")
print("Prediction:", prediction)
print("Confidence:", confidence_bps / 100, "%")
print("Simulation succeeded.")
print("Estimated gas:", gas_estimate)

In [ ]:
from getpass import getpass
from eth_account import Account

# Confirm the current contract owner
owner = contract.functions.owner().call()

# Enter your test wallet private key privately
private_key = getpass("Enter test wallet private key (hidden): ")
account = Account.from_key(private_key)

# Ensure the signing wallet is authorized
if account.address.lower() != owner.lower():
    private_key = None
    raise ValueError("Wallet is not the contract owner. No transaction sent.")

# Build the registration transaction
tx = contract.functions.registerImage(
    image_id,
    sha256_bytes,
    prediction,
    confidence_bps
).build_transaction({
    "from": account.address,
    "nonce": w3.eth.get_transaction_count(account.address, "pending"),
    "chainId": 11155111,
    "gas": 700000,
    "gasPrice": w3.eth.gas_price
})

# Sign and submit
signed_tx = account.sign_transaction(tx)
tx_hash = w3.eth.send_raw_transaction(signed_tx.raw_transaction)

# Remove references to the private key as soon as possible
private_key = None
signed_tx = None
account = None

print("Transaction submitted:", tx_hash.hex())
print("Waiting for confirmation...")

receipt = w3.eth.wait_for_transaction_receipt(tx_hash, timeout=180)

print("Transaction status:", receipt.status)
print("Gas used:", receipt.gasUsed)
print("Block number:", receipt.blockNumber)

if receipt.status != 1:
    raise RuntimeError("Registration transaction failed.")

In [ ]:
record = contract.functions.getImageRecord("IMG_0002").call()

print("Record exists:", record[4])
print("Stored hash:", "0x" + record[0].hex())
print("Prediction:", record[1])
print("Confidence:", record[2] / 100, "%")
print("Timestamp:", record[3])

In [ ]:
# Verify the registered certificate using its original hash
original_hash = bytes.fromhex(
    "ebb5bb8fd45137c2b58ae25c32a42ef03c61f3119bc37cfc9d7dd018a2635b24"
)

result = contract.functions.verifyImage(
    "IMG_0002",
    original_hash
).call()

print("Registered:", result[0])
print("Hash matches:", result[1])

if result[0] and result[1]:
    print("Image verified - Hash matches blockchain record")
else:
    print("Verification failed - Hash mismatch")

In [ ]:
import hashlib
from google.colab import files

# 1. Upload the certificate to verify
uploaded = files.upload()
filename = next(iter(uploaded))

# 2. Calculate SHA-256 from the exact uploaded file bytes
with open(filename, "rb") as f:
    uploaded_hash = hashlib.sha256(f.read()).digest()

print("Uploaded filename:", filename)
print("New SHA-256:", uploaded_hash.hex())

# 3. Choose the ID of the trusted original certificate
# This example compares against your registered IMG_0002 record.
image_id = "IMG_0002"

# 4. Ask the deployed smart contract to verify the hash
registered, hash_matches = contract.functions.verifyImage(
    image_id,
    uploaded_hash
).call()

# 5. Display the blockchain verification result
print("\nRegistered record:", registered)
print("Hash matches:", hash_matches)

if not registered:
    print("No registered original exists for this certificate ID.")
elif hash_matches:
    print("Image verified - Hash matches blockchain record")
else:
    print("Verification failed - Hash mismatch")

In [ ]:
#-----Connecting Blockchain and DL-------

In [ ]:
# SECTION 1: Load the trained EfficientNet-B0 model
# Used to classify uploaded certificates as Original or Tampered.
import hashlib
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
from pathlib import Path

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_PATH = "/content/drive/MyDrive/efficientnet_b0_certificate_tampering_FINAL.pth"

# Load the trained model
model = models.efficientnet_b0(weights=None)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)

checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(DEVICE)
model.eval()

CLASS_NAMES = {0: "Original", 1: "Tampered"}

inference_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

def predict_certificate(image_path):
    image = Image.open(image_path).convert("RGB")
    image_tensor = inference_transform(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        outputs = model(image_tensor)
        probabilities = torch.softmax(outputs, dim=1)
        confidence, predicted_class = probabilities.max(dim=1)

    return (
        CLASS_NAMES[predicted_class.item()],
        confidence.item() * 100
    )

print("Model loaded and prediction function ready!")

In [ ]:
#------Removing hardcoded integration
# TEST SETUP: Select a certificate, calculate its SHA-256 hash,
# and run the AI prediction.

In [ ]:
from pathlib import Path

ORIGINAL_DIR = Path(
    "/content/drive/MyDrive/capstone_dataset creation/"
    "capstone_dataset creation/original"
)

image_files = sorted(
    p for p in ORIGINAL_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in [".jpg", ".jpeg", ".png"]
)

for i, path in enumerate(image_files):
    print(i, path.name)

In [ ]:
# Select a certificate by its number from the list above
selected_index = 0

TEST_IMAGE = str(image_files[selected_index])

# Calculate SHA-256 from the exact file bytes
with open(TEST_IMAGE, "rb") as f:
    sha256_hex = hashlib.sha256(f.read()).hexdigest()

# Run the AI prediction
prediction, confidence = predict_certificate(TEST_IMAGE)

print("Filename:", Path(TEST_IMAGE).name)
print("SHA-256:", sha256_hex)
print("Prediction:", prediction)
print(f"Confidence: {confidence:.2f}%")
print("Confidence for contract:", round(confidence * 100), "basis points")

In [ ]:
from pathlib import Path

HASH_DIR = Path("/content/drive/MyDrive/capstone_blockchain_files")

for name in ["image_hashes.csv", "image_records.json"]:
    path = HASH_DIR / name
    print(name, "FOUND" if path.is_file() else "NOT FOUND")


In [ ]:
import pandas as pd

HASH_CSV = "/content/drive/MyDrive/capstone_blockchain_files/image_hashes.csv"

df = pd.read_csv(HASH_CSV)

print("Columns:", df.columns.tolist())
print(df.to_string(index=False))

In [ ]:
# TEST SECTION: Match a selected certificate to its CSV ID,
# then verify its hash against the blockchain.
# Find the certificate's ID from your existing CSV
matches = df[df["filename"] == Path(TEST_IMAGE).name]

if len(matches) != 1:
    print("Could not find exactly one matching certificate in CSV.")
else:
    image_id = matches.iloc[0]["image_id"]
    recorded_hash = matches.iloc[0]["sha256"]

    print("Image ID:", image_id)
    print("CSV hash matches uploaded file:", recorded_hash == sha256_hex)

    result = contract.functions.verifyImage(
        image_id,
        bytes.fromhex(sha256_hex)
    ).call()

    print("Registered on blockchain:", result[0])
    print("Hash matches blockchain:", result[1])
    print("AI Prediction:", prediction)
    print(f"AI Confidence: {confidence:.2f}%")

In [ ]:
# REGISTRATION PRE-CHECK SECTION
# Validates each CSV hash against the original file and blockchain.
# Skips certificates already registered with the correct hash.
# Prepares only missing certificates for registration.
import hashlib
from pathlib import Path

pending_records = []

for _, row in df.iterrows():
    image_id = str(row["image_id"]).strip()
    filename = str(row["filename"]).strip()
    expected_hash = str(row["sha256"]).strip().lower()

    image_path = ORIGINAL_DIR / filename

    if not image_path.is_file():
        raise FileNotFoundError(f"Missing original image: {filename}")

    with open(image_path, "rb") as f:
        actual_hash = hashlib.sha256(f.read()).hexdigest()

    if actual_hash != expected_hash:
        raise ValueError(f"Hash mismatch in CSV for {filename}")

    record = contract.functions.getImageRecord(image_id).call()
    exists = record[4]

    if exists:
        stored_hash = record[0].hex().lower().removeprefix("0x")
        actual_hash = actual_hash.lower().removeprefix("0x")

        if stored_hash != actual_hash:
            raise ValueError(
                f"{image_id} already exists with a different hash. "
                "Stopping to avoid overwriting or misidentifying it."
            )

        print(f"{image_id}: already registered correctly; skipping.")
        continue

    prediction, confidence = predict_certificate(str(image_path))

    pending_records.append({
        "image_id": image_id,
        "filename": filename,
        "sha256": actual_hash,
        "prediction": prediction,
        "confidence_bps": round(confidence * 100)
    })

print("\nRecords ready for registration:", len(pending_records))

for record in pending_records:
    print(
        record["image_id"],
        "|", record["filename"],
        "|", record["prediction"],
        "|", record["confidence_bps"] / 100, "%"
    )

In [ ]:
# REGISTRATION SECTION
# Run only when new certificates need to be registered.
# Do not rerun unnecessarily; existing records are already on-chain.

from getpass import getpass

if not pending_records:
    print("No pending records to register.")
else:
    private_key = getpass("Enter the contract owner's private key (input stays hidden): ")
    account = w3.eth.account.from_key(private_key)

    owner_address = contract.functions.owner().call()

    if account.address.lower() != owner_address.lower():
        private_key = None
        raise ValueError("This wallet is not the contract owner. No transactions sent.")

    balance = w3.eth.get_balance(account.address)
    print("Owner wallet verified:", account.address)
    print("Sepolia ETH balance:", w3.from_wei(balance, "ether"))

    if balance == 0:
        private_key = None
        raise ValueError("Wallet has no Sepolia ETH for transaction gas.")

    registered_results = []

    try:
        for item in pending_records:
            image_id = item["image_id"]

            # Re-check the chain before every transaction.
            existing = contract.functions.getImageRecord(image_id).call()

            if existing[4]:
                print(image_id, "is already registered; skipping.")
                continue

            tx = contract.functions.registerImage(
                image_id,
                bytes.fromhex(item["sha256"]),
                item["prediction"],
                item["confidence_bps"]
            ).build_transaction({
                "from": account.address,
                "nonce": w3.eth.get_transaction_count(account.address, "pending"),
                "chainId": 11155111,
                "gas": int(
                    contract.functions.registerImage(
                        image_id,
                        bytes.fromhex(item["sha256"]),
                        item["prediction"],
                        item["confidence_bps"]
                    ).estimate_gas({"from": account.address}) * 1.2
                ),
                "gasPrice": w3.eth.gas_price
            })

            signed_tx = w3.eth.account.sign_transaction(tx, private_key)
            tx_hash = w3.eth.send_raw_transaction(signed_tx.raw_transaction)
            receipt = w3.eth.wait_for_transaction_receipt(tx_hash)

            if receipt.status != 1:
                raise RuntimeError(f"{image_id} transaction failed: {tx_hash.hex()}")

            registered_results.append((image_id, tx_hash.hex()))
            print(image_id, "registered | TX:", tx_hash.hex())

    finally:
        private_key = None

    print("\nRegistration transactions completed:", len(registered_results))


In [ ]:
# VERIFICATION SECTION: Check all CSV certificate records against the blockchain.
# Displays registration status, hash match, and stored AI prediction.

for _, row in df.iterrows():
    image_id = str(row["image_id"]).strip()
    expected_hash = str(row["sha256"]).strip().lower()

    record = contract.functions.getImageRecord(image_id).call()
    exists = record[4]
    stored_hash = record[0].hex().lower().removeprefix("0x")

    print(
        image_id,
        "| Registered:", exists,
        "| Hash matches:", exists and stored_hash == expected_hash,
        "| Prediction:", record[1] if exists else "N/A"
    )


In [ ]:
# TEST SECTION: Verify an unchanged original certificate.
# Expected result: Registered = True, Hash matches = True.

image_id = "IMG_0001"
image_path = str(ORIGINAL_DIR / "image_original_1.jpeg")

with open(image_path, "rb") as f:
    uploaded_hash = hashlib.sha256(f.read()).digest()

registered, hash_matches = contract.functions.verifyImage(
    image_id, uploaded_hash
).call()

print("Registered:", registered)
print("Hash matches:", hash_matches)

In [ ]:
# TEST SECTION: Verify blockchain detection of a modified certificate.
# Creates a temporary modified copy without changing the original file.
# Expected result: Registered = True, Hash matches = False.

from PIL import Image
import hashlib
import tempfile
from pathlib import Path
from PIL import Image
import hashlib
import tempfile
from pathlib import Path

original_path = ORIGINAL_DIR / "image_original_1.jpeg"

# Create a temporary modified copy; don't overwrite the original.
with Image.open(original_path) as img:
    modified = img.convert("RGB")
    pixel = modified.getpixel((0, 0))
    modified.putpixel((0, 0), tuple(255 - value for value in pixel))

temp_path = Path(tempfile.gettempdir()) / "test_tampered_certificate.png"
modified.save(temp_path, format="PNG")

with open(temp_path, "rb") as f:
    tampered_hash = hashlib.sha256(f.read()).digest()

registered, hash_matches = contract.functions.verifyImage(
    "IMG_0001", tampered_hash
).call()

print("Registered:", registered)
print("Hash matches:", hash_matches)
print("Test file:", temp_path)


In [ ]:
import hashlib
from pathlib import Path

# FINAL INTEGRATION FUNCTION
# Predicts the certificate class, calculates its SHA-256 hash,
# and checks the uploaded file against the blockchain record.

def verify_certificate_pipeline(image_path, image_id):
    image_path = Path(image_path)

    if not image_path.is_file():
        raise FileNotFoundError(f"Certificate not found: {image_path}")

    # 1. Deep-learning prediction
    prediction, confidence = predict_certificate(str(image_path))

    # 2. SHA-256 of the exact uploaded file bytes
    with open(image_path, "rb") as f:
        file_hash = hashlib.sha256(f.read()).hexdigest()

    # 3. Verify against the blockchain record
    uploaded_hash = bytes.fromhex(file_hash)

    registered, hash_matches = contract.functions.verifyImage(
        image_id, uploaded_hash
    ).call()

    # 4. Retrieve the trusted record, if registered
    record = contract.functions.getImageRecord(image_id).call()

    result = {
        "image_id": image_id,
        "prediction": prediction,
        "confidence": round(confidence, 2),
        "sha256": file_hash,
        "registered": registered,
        "hash_matches": hash_matches,
        "blockchain_prediction": record[1] if registered else None
    }

    return result


In [ ]:
result = verify_certificate_pipeline(
    ORIGINAL_DIR / "image_original_1.jpeg",
    "IMG_0001"
)

for key, value in result.items():
    print(f"{key}: {value}")


In [ ]:
result = verify_certificate_pipeline(
    "/tmp/test_tampered_certificate.png",
    "IMG_0001"
)

for key, value in result.items():
    print(f"{key}: {value}")
